# LAB 4 — Benchmark: N=1, N=3, N=5 Queries
## Aula 7 · MBA RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Medir sistematicamente o impacto de N (número de queries/sub-queries) sobre recall, latência e consumo de tokens no pipeline de Multi-Query RAG. Produzir gráficos de análise e uma recomendação fundamentada do valor ótimo de N.

**Tempo estimado:** 30 minutos

---
**Checklist de entrega:**
- [ ] Tabela benchmark com recall, latência e tokens para N=1, N=3, N=5
- [ ] Gráfico: recall × N
- [ ] Gráfico: latência × N
- [ ] Análise de trade-off com recomendação justificada

In [ ]:
!pip install -q langchain langchain-openai sentence-transformers opensearch-py ragas pandas matplotlib
print('✅ Dependências instaladas!')

In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer
from opensearchpy import OpenSearch

VLLM_BASE_URL = os.getenv('VLLM_BASE_URL', 'http://localhost:8000/v1')
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
INDEX_NAME = 'corpus_juridico_aula7'

llm = ChatOpenAI(
    base_url=VLLM_BASE_URL, api_key='dummy',
    model='meta-llama/Llama-3.1-8B-Instruct',
    temperature=0.5
)
embeddings = SentenceTransformer('BAAI/bge-m3')
os_client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': 9200}],
    http_auth=('admin', os.getenv('OPENSEARCH_PASS', 'Admin123!')),
    use_ssl=False, verify_certs=False
)
print('✅ Ambiente pronto!')

## 2. Funções Auxiliares

In [ ]:
PROMPT_VARIACOES = PromptTemplate(
    input_variables=['question', 'n'],
    template="""Gere exatamente {n} versões alternativas da pergunta jurídica abaixo.
Retorne APENAS as perguntas, uma por linha.
Pergunta: {question}
Versões:"""
)

def gerar_variacoes(query: str, n: int) -> tuple[list[str], int]:
    """Gera N variações e retorna (variações, tokens_usados)."""
    if n == 1:
        return [query], 0  # N=1 = Naive RAG, sem chamada LLM
    
    response = llm.invoke(PROMPT_VARIACOES.format(question=query, n=n-1))
    variacoes = [v.strip() for v in response.content.strip().split('\n') if v.strip()]
    all_queries = [query] + variacoes[:n-1]
    
    # Estimar tokens (simples: len palavras × 1.3)
    tokens = int(sum(len(q.split()) for q in all_queries) * 1.3)
    return all_queries, tokens

def retrieve_multiple(queries: list[str], k: int = 5) -> list[dict]:
    """Retrieval sequencial (síncrono) para lista de queries, com deduplicação."""
    todos_docs = {}
    
    for query in queries:
        q_vec = embeddings.encode(query, normalize_embeddings=True).tolist()
        resp = os_client.search(
            index=INDEX_NAME,
            body={
                'size': k,
                'query': {'knn': {'embedding': {'vector': q_vec, 'k': k}}},
                '_source': ['content', 'source']
            }
        )
        for h in resp['hits']['hits']:
            doc_id = h['_id']
            if doc_id not in todos_docs:
                todos_docs[doc_id] = {
                    'id': doc_id,
                    'score': h['_score'],
                    'content': h['_source'].get('content', '')
                }
    
    return list(todos_docs.values())

def calcular_recall_heuristico(docs: list[dict], ground_truth_ids: list[str]) -> float:
    """Calcula recall: % dos documentos relevantes recuperados."""
    if not ground_truth_ids:
        return 0.0
    recuperados = {d['id'] for d in docs}
    encontrados = len(set(ground_truth_ids) & recuperados)
    return encontrados / len(ground_truth_ids)

print('✅ Funções auxiliares definidas!')

## 3. Dataset de Benchmark

In [ ]:
# Dataset de benchmark: queries com documentos relevantes conhecidos
# Em produção, este ground truth seria construído manualmente (Lab da Aula 5)
# Aqui usamos IDs fictícios para demonstração do framework

benchmark_queries = [
    {
        'query': 'Quais são os requisitos para decretação de prisão preventiva?',
        'ground_truth_ids': ['doc_001', 'doc_002', 'doc_003', 'doc_007'],
        'keywords': ['preventiva', 'requisitos', 'art. 312', 'cautelar']
    },
    {
        'query': 'Como funciona a medida protetiva de urgência na Lei Maria da Penha?',
        'ground_truth_ids': ['doc_010', 'doc_011', 'doc_015'],
        'keywords': ['medida protetiva', 'Maria da Penha', 'violência doméstica']
    },
    {
        'query': 'Quais são os direitos do preso durante o interrogatório policial?',
        'ground_truth_ids': ['doc_020', 'doc_021', 'doc_022', 'doc_025'],
        'keywords': ['interrogatório', 'direitos', 'silêncio', 'advogado']
    },
    {
        'query': 'Quando é cabível habeas corpus e qual é o procedimento?',
        'ground_truth_ids': ['doc_030', 'doc_031', 'doc_035'],
        'keywords': ['habeas corpus', 'liberdade', 'ilegalidade', 'coação']
    },
    {
        'query': 'Como é calculada a dosimetria da pena no sistema trifásico?',
        'ground_truth_ids': ['doc_040', 'doc_041', 'doc_042'],
        'keywords': ['dosimetria', 'pena-base', 'agravantes', 'art. 68']
    }
]

print(f'Dataset de benchmark: {len(benchmark_queries)} queries')
for i, q in enumerate(benchmark_queries, 1):
    print(f'  Q{i}: {q["query"][:60]}...')

## 4. Benchmark Principal: N=1, N=3, N=5

In [ ]:
valores_N = [1, 3, 5]
K_RETRIEVAL = 5  # documentos por query

resultados_benchmark = []

for N in valores_N:
    print(f'\n{"="*50}')
    print(f'TESTANDO N = {N} {"(Naive RAG)" if N==1 else "queries"}')
    print(f'Esperado: {N} × {K_RETRIEVAL} = {N*K_RETRIEVAL} docs (antes dedup)')
    print('='*50)
    
    recalls = []
    latencias = []
    tokens_totais = []
    
    for q_data in benchmark_queries:
        query = q_data['query']
        ground_truth = q_data['ground_truth_ids']
        keywords = q_data['keywords']
        
        t_inicio = time.time()
        
        # Gerar variações
        queries_n, tokens_gen = gerar_variacoes(query, N)
        
        # Retrieval
        docs = retrieve_multiple(queries_n, k=K_RETRIEVAL)
        
        t_latencia = time.time() - t_inicio
        
        # Métricas
        # Recall: baseado em keywords (heurística quando não há ground truth anotado)
        recall_keywords = sum(
            1 for doc in docs
            if any(kw.lower() in doc['content'].lower() for kw in keywords)
        ) / max(len(docs), 1)
        
        recalls.append(recall_keywords)
        latencias.append(t_latencia)
        # Tokens: geração de variações + retrieval (estimativa)
        tokens_retrieval = N * K_RETRIEVAL * 50  # ~50 tokens por doc processado
        tokens_totais.append(tokens_gen + tokens_retrieval)
        
        print(f'  Q: {query[:45]}... | Recall={recall_keywords:.0%} | {t_latencia:.2f}s | {len(docs)} docs únicos')
    
    resultados_benchmark.append({
        'N': N,
        'recall_medio': np.mean(recalls),
        'recall_std': np.std(recalls),
        'latencia_media_s': np.mean(latencias),
        'latencia_std_s': np.std(latencias),
        'tokens_medios': int(np.mean(tokens_totais)),
        'custo_estimado_1000q': f'${np.mean(tokens_totais) * 0.00015 * 1000:.2f}'  # ~$0.15/1M tokens
    })

df_benchmark = pd.DataFrame(resultados_benchmark)
print('\n=== TABELA FINAL DO BENCHMARK ===')
print(df_benchmark.to_string(index=False))

## 5. Visualização do Benchmark

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Benchmark Multi-Query RAG: Impacto de N\nCorpus Jurídico | Google Colab T4', fontsize=13)

x = df_benchmark['N'].values

# Gráfico 1: Recall × N
axes[0].plot(x, df_benchmark['recall_medio'], 'o-', color='#2ecc71', linewidth=2, markersize=8)
axes[0].fill_between(x,
    df_benchmark['recall_medio'] - df_benchmark['recall_std'],
    df_benchmark['recall_medio'] + df_benchmark['recall_std'],
    alpha=0.2, color='#2ecc71')
axes[0].set_title('Recall × N', fontsize=12)
axes[0].set_xlabel('N (número de queries)')
axes[0].set_ylabel('Recall (keywords)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['N=1\n(Naive)', 'N=3', 'N=5'])
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1.05)
for xi, yi in zip(x, df_benchmark['recall_medio']):
    axes[0].annotate(f'{yi:.0%}', (xi, yi), textcoords='offset points', xytext=(5, 5))

# Gráfico 2: Latência × N
axes[1].bar(range(len(x)), df_benchmark['latencia_media_s'],
            yerr=df_benchmark['latencia_std_s'],
            color='#e74c3c', alpha=0.8, capsize=5)
axes[1].set_title('Latência Média × N', fontsize=12)
axes[1].set_xlabel('N (número de queries)')
axes[1].set_ylabel('Latência (segundos)')
axes[1].set_xticks(range(len(x)))
axes[1].set_xticklabels(['N=1\n(Naive)', 'N=3', 'N=5'])
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(df_benchmark['latencia_media_s']):
    axes[1].text(i, v + 0.05, f'{v:.2f}s', ha='center', va='bottom')

# Gráfico 3: Custo estimado × N (tokens)
axes[2].bar(range(len(x)), df_benchmark['tokens_medios'],
            color='#9b59b6', alpha=0.8)
axes[2].set_title('Tokens Médios × N\n(Geração + Retrieval)', fontsize=12)
axes[2].set_xlabel('N (número de queries)')
axes[2].set_ylabel('Tokens por Query')
axes[2].set_xticks(range(len(x)))
axes[2].set_xticklabels(['N=1\n(Naive)', 'N=3', 'N=5'])
axes[2].grid(axis='y', alpha=0.3)
for i, (tokens, custo) in enumerate(zip(df_benchmark['tokens_medios'], df_benchmark['custo_estimado_1000q'])):
    axes[2].text(i, tokens + 10, f'{tokens}\n({custo}/1k)', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('benchmark_multi_query.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Benchmark salvo como benchmark_multi_query.png')

## 6. Análise de Trade-off e Recomendação

In [ ]:
print('=== ANÁLISE DE TRADE-OFF: N=1 vs N=3 vs N=5 ===')
print()

recall_n1 = df_benchmark[df_benchmark['N']==1]['recall_medio'].values[0]
recall_n3 = df_benchmark[df_benchmark['N']==3]['recall_medio'].values[0]
recall_n5 = df_benchmark[df_benchmark['N']==5]['recall_medio'].values[0]

lat_n1 = df_benchmark[df_benchmark['N']==1]['latencia_media_s'].values[0]
lat_n3 = df_benchmark[df_benchmark['N']==3]['latencia_media_s'].values[0]
lat_n5 = df_benchmark[df_benchmark['N']==5]['latencia_media_s'].values[0]

print(f'RECALL:')
print(f'  N=1 (Naive): {recall_n1:.0%} (baseline)')
print(f'  N=3:         {recall_n3:.0%} (+{(recall_n3-recall_n1):.0%} vs N=1)')
print(f'  N=5:         {recall_n5:.0%} (+{(recall_n5-recall_n1):.0%} vs N=1)')
print()
print(f'LATÊNCIA:')
print(f'  N=1 (Naive): {lat_n1:.2f}s')
print(f'  N=3:         {lat_n3:.2f}s (×{lat_n3/lat_n1:.1f} vs N=1)')
print(f'  N=5:         {lat_n5:.2f}s (×{lat_n5/lat_n1:.1f} vs N=1)')
print()
print(f'RETORNO MARGINAL DE RECALL:')
print(f'  N=1→N=3: {(recall_n3-recall_n1):.0%} de ganho')
print(f'  N=3→N=5: {(recall_n5-recall_n3):.0%} de ganho (menor retorno)')

print()
print('='*50)
print('RECOMENDAÇÃO:')
print('='*50)

# Calcular ponto ótimo (melhor recall/latência trade-off)
# Eficiência = ganho de recall por unidade de latência adicional
ganho_recall_1_3 = recall_n3 - recall_n1
custo_latencia_1_3 = lat_n3 - lat_n1
eficiencia_1_3 = ganho_recall_1_3 / max(custo_latencia_1_3, 0.001)

ganho_recall_3_5 = recall_n5 - recall_n3
custo_latencia_3_5 = lat_n5 - lat_n3
eficiencia_3_5 = ganho_recall_3_5 / max(custo_latencia_3_5, 0.001)

print(f'  Eficiência N=1→N=3: {eficiencia_1_3:.3f} recall/segundo')
print(f'  Eficiência N=3→N=5: {eficiencia_3_5:.3f} recall/segundo')
print()
print('  → N=3 oferece o melhor equilíbrio entre recall e latência')
print('    para a maioria dos casos jurídicos analisados.')
print()
print('  Exceções:')
print('  • Usar N=1 se latência < 1s for crítica (chatbot em tempo real)')
print('  • Usar N=5 se o corpus for muito técnico e latência for tolerável')

## 7. Reflexão Final

Escreva abaixo sua análise:

### Sua Recomendação de N para o Projeto Final

Com base nos resultados deste benchmark, qual valor de N você escolheria para o projeto final do curso (chatbot jurídico para uma delegacia de polícia)? Justifique considerando:
1. O perfil dos usuários (policiais, nem sempre com vocabulário jurídico técnico)
2. A criticidade das respostas (investigações reais)
3. O volume esperado de queries por dia
4. Os recursos computacionais disponíveis (GPU local)

> **Sua resposta aqui:**

In [ ]:
print('=== CHECKLIST DE ENTREGA — LAB 4 ===')
checklist = [
    'Benchmark executado para N=1, N=3 e N=5',
    'Tabela com recall, latência e tokens documentada',
    'Gráfico recall × N gerado e salvo',
    'Gráfico latência × N gerado e salvo',
    'Análise de trade-off com recomendação justificada escrita',
    'Reflexão sobre o projeto final completada'
]
for item in checklist:
    print(f'  [ ] {item}')
print('\n✅ Lab 4 concluído! Prossiga para o LAB 5 — LangFuse.')